In [2]:
%cd ..

c:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions


In [3]:
from dotenv import load_dotenv

load_dotenv()


True

In [4]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [5]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [6]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/finance-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "finance_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/finance_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/finance_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    return False


In [7]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_csv_and_normalize_columns(
    file_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_csv(file_path, header=None, encoding="utf-8", dtype=str)
    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(str(col).strip())

            # 2️⃣ fallback snake_case
            if base is None:
                base = snake_case(col)
                print(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)
    df = df.fillna("")

    return df

def read_folder_and_union_csv(
    folder_path: str,
    mapping: dict,
    header_row=0,
    drop_rows=0,
    encoding="utf-8"
):
    dfs = []
    folder = Path(folder_path)

    files = sorted(folder.glob("*.csv"))
    if not files:
        raise ValueError("❌ Không tìm thấy file CSV nào trong folder")

    for file in files:
        df = read_csv_and_normalize_columns(
            file_path=file,
            mapping=mapping,
            drop_rows=drop_rows,
            header_row=header_row
        )

        # trace file nguồn
        df["source_file"] = file.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [8]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=0
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [15]:
import pandas as pd

COLUMN_DICT_FINANCE_ACTUAL_COST = {
    "Ngày": "report_date",
    "Tháng": "report_month",
    "Sản phẩm/ Dịch vụ": "old_category",
    "Khoản mục SPDV": "cost_group",
    "Triệu đồng": "base_currency_amount",
    "Mã SPDV": "category_code",
    "Mã KM phí": "km_cost_code"
}

filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\chiphi_spdv\actual_cost_2025.xlsx"
# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\chiphi_spdv\chi_phi_spdv_T02_2026.xlsx"
resource_name = "actual_cost"

df = pd.read_excel(filename, sheet_name="Sheet1", dtype=str)

# -------------------- CLEAN HEADER --------------------
df.columns = df.columns.str.strip()
# df["report_year"] = pd.to_numeric(df["report_month"], errors="coerce")
# df["report_month"] = pd.to_numeric(df["report_month"], errors="coerce")
df["base_currency_amount"] = pd.to_numeric(df["base_currency_amount"], errors="coerce")


# -------------------- SELECT + RENAME --------------------
# df = df[[col for col in df.columns if col in COLUMN_DICT_FINANCE_ACTUAL_COST]]
df = df.rename(columns=COLUMN_DICT_FINANCE_ACTUAL_COST)

df["source_file"] = filename.split("\\")[-1]

# -------------------- FINAL CLEAN --------------------
df = df.fillna("")

print(df.columns)
print(df.dtypes)

df.head(2)

Index(['report_date', 'report_year', 'report_month', 'old_category',
       'category_code', 'cost_group', 'base_currency_amount', 'currency_code',
       'territory_name', 'product_category', 'unit_level_1', 'source_file',
       'crawled_at_ts'],
      dtype='object')
report_date              object
report_year              object
report_month             object
old_category             object
category_code            object
cost_group               object
base_currency_amount    float64
currency_code            object
territory_name           object
product_category         object
unit_level_1             object
source_file              object
crawled_at_ts            object
dtype: object


,report_date,report_year,report_month,old_category,category_code,cost_group,base_currency_amount,currency_code,territory_name,product_category,unit_level_1,source_file,crawled_at_ts
0,2025-01-31 00:00:00,2025,1,BRG,VCS022,Chi phí khấu hao tài sản cố định,23239253.19,VND,Miền bắc,BRG,TTSP TELCO,actual_cost_2025.xlsx,
1,2025-01-31 00:00:00,2025,1,Cloudrity,VCS013,Chi phí khấu hao tài sản cố định,93840769.87,VND,Miền bắc,Cloudrity,TTSP TELCO,actual_cost_2025.xlsx,


In [16]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

✅ Done: actual_cost.json created
Start crawl :  actual_cost
actual_cost
Replace Upload  s3a://vcs-raw/finance-raw/actual_cost ./tmp/data/finance_raw/actual_cost/data_actual_cost_20260407_155901.parquet
bucket=vcs-raw , key=finance-raw/actual_cost
objects_to_delete=[]
bucket=vcs-raw , key=finance-raw/actual_cost/data_actual_cost_20260407_155901.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/actual_cost
Uploaded SQL definition


False

In [19]:
import pandas as pd

COLUMN_DICT_FINANCE_ACTUAL_COST = {
    "Ngày": "report_date",
    "Tháng": "report_month",
    "Sản phẩm/ Dịch vụ": "old_category",
    "Khoản mục SPDV": "cost_group",
    "Triệu đồng": "base_currency_amount",
    "Mã SPDV": "category_code",
    "Mã KM phí": "km_cost_code"
}

# filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\chiphi_spdv\chi_phi_spdv_T01_2026.xlsx"
filename = r"C:\Users\namtv40\Documents\AI Chatbot\finance-data\chiphi_spdv\chi_phi_spdv_T02_2026.xlsx"
resource_name = "actual_cost"

df = pd.read_excel(filename, sheet_name="Sheet1", dtype=str)

# -------------------- CLEAN HEADER --------------------
df.columns = df.columns.str.strip()

# -------------------- SELECT + RENAME --------------------
df = df[[col for col in df.columns if col in COLUMN_DICT_FINANCE_ACTUAL_COST]]
df = df.rename(columns=COLUMN_DICT_FINANCE_ACTUAL_COST)

# -------------------- TYPE CAST --------------------
df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")

df["report_year"] = df["report_date"].dt.year
df["report_month"] = pd.to_numeric(df["report_month"], errors="coerce")

df["base_currency_amount"] = pd.to_numeric(df["base_currency_amount"], errors="coerce")

# triệu đồng → đồng
df["base_currency_amount"] = df["base_currency_amount"] * 1_000_000

# -------------------- ADD COLUMN --------------------
df["currency_code"] = "VND"

# territory logic
df["territory_name"] = df["old_category"].str.contains("HCM", case=False, na=False)
df["territory_name"] = df["territory_name"].map({
    True: "Miền Nam",
    False: "Miền Bắc"
})

df["product_category"] = ""
df["unit_level_1"] = ""

df["source_file"] = filename.split("\\")[-1]
df["report_date"] = df["report_date"].dt.strftime("%Y-%m-%d")

# -------------------- FINAL CLEAN --------------------
df = df.fillna("")

print(df.columns)
print(df.dtypes)

df.head(2)

Index(['report_date', 'report_month', 'old_category', 'cost_group',
       'base_currency_amount', 'category_code', 'km_cost_code', 'report_year',
       'currency_code', 'territory_name', 'product_category', 'unit_level_1',
       'source_file'],
      dtype='object')
report_date              object
report_month              int64
old_category             object
cost_group               object
base_currency_amount    float64
category_code            object
km_cost_code             object
report_year               int32
currency_code            object
territory_name           object
product_category         object
unit_level_1             object
source_file              object
dtype: object


,report_date,report_month,old_category,cost_group,base_currency_amount,category_code,km_cost_code,report_year,currency_code,territory_name,product_category,unit_level_1,source_file
0,2026-02-28,2,MSS,Chi phí Nhân công & OS trực tiếp,3.609130e+09,VCS001,VCSKMSPCP01,2026,VND,Miền Bắc,,,chi_phi_spdv_T02_2026.xlsx
1,2026-02-28,2,MSS,"Chi phí phần cứng, phần mềm mua/thuê ngoài",0.000000e+00,VCS001,VCSKMSPCP08,2026,VND,Miền Bắc,,,chi_phi_spdv_T02_2026.xlsx


In [20]:
# 6. Chuyển sang JSON
records = df.to_dict(orient="records")
with open(resource_name +".json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")

fetch_resource_name_freshwork(resource_name, records, crawl_mode="modified_and_new")

✅ Done: actual_cost.json created
Start crawl :  actual_cost
actual_cost
Add Upload  s3a://vcs-raw/finance-raw/actual_cost ./tmp/data/finance_raw/actual_cost/data_actual_cost_20260407_155916.parquet
bucket=vcs-raw , key=finance-raw/actual_cost/data_actual_cost_20260407_155916.parquet
Uploaded parquet to s3a://vcs-raw/finance-raw/actual_cost
Uploaded SQL definition


False